In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# MRI Data Pipeline

Downloads brain MRI scans from OpenNeuro (ds000221, ds000114, ds002330) and preprocesses them into 2D PNG slices for model training.

Labels: T1w · T2w · FLAIR · DWI · BOLD : 50 volumes per class, 5 slices per volume - 1250 PNGs total.

In [ ]:
!pip install awscli


## Data Download

Pulling NIfTI files from OpenNeuro S3 using the AWS CLI.

In [ ]:
!aws s3 sync s3://openneuro.org/ds000221 ./ds000221 \
  --no-sign-request \
  --exclude "*" \
  --include "sub-010002/ses-01/anat/*acq-mp2rage*T1w.nii.gz" \
  --include "sub-010003/ses-01/anat/*acq-mp2rage*T1w.nii.gz" \
  --include "sub-010004/ses-01/anat/*acq-mp2rage*T1w.nii.gz" \
  --include "sub-010005/ses-01/anat/*acq-mp2rage*T1w.nii.gz" \
  --include "sub-010006/ses-01/anat/*acq-mp2rage*T1w.nii.gz" \
  --include "sub-010007/ses-01/anat/*acq-mp2rage*T1w.nii.gz" \
  --include "sub-010010/ses-01/anat/*acq-mp2rage*T1w.nii.gz" \
  --include "sub-010012/ses-01/anat/*acq-mp2rage*T1w.nii.gz" \
  --include "sub-010015/ses-01/anat/*acq-mp2rage*T1w.nii.gz" \
  --include "sub-010016/ses-01/anat/*acq-mp2rage*T1w.nii.gz" \
  --include "sub-010017/ses-01/anat/*acq-mp2rage*T1w.nii.gz" \
  --include "sub-010019/ses-01/anat/*acq-mp2rage*T1w.nii.gz" \
  --include "sub-010020/ses-01/anat/*acq-mp2rage*T1w.nii.gz" \
  --include "sub-010021/ses-01/anat/*acq-mp2rage*T1w.nii.gz" \
  --include "sub-010022/ses-01/anat/*acq-mp2rage*T1w.nii.gz" \
  --include "sub-010023/ses-01/anat/*acq-mp2rage*T1w.nii.gz" \
  --include "sub-010024/ses-01/anat/*acq-mp2rage*T1w.nii.gz" \
  --include "sub-010026/ses-01/anat/*acq-mp2rage*T1w.nii.gz" \
  --include "sub-010027/ses-01/anat/*acq-mp2rage*T1w.nii.gz" \
  --include "sub-010028/ses-01/anat/*acq-mp2rage*T1w.nii.gz"

In [ ]:
!aws s3 sync s3://openneuro.org/ds002330 ./ds002330 \
  --no-sign-request \
  --exclude "*" \
  --include "sub-01/anat/*T1w.nii.gz" \
  --include "sub-02/anat/*T1w.nii.gz" \
  --include "sub-03/anat/*T1w.nii.gz" \
  --include "sub-04/anat/*T1w.nii.gz" \
  --include "sub-05/anat/*T1w.nii.gz" \
  --include "sub-06/anat/*T1w.nii.gz" \
  --include "sub-07/anat/*T1w.nii.gz" \
  --include "sub-08/anat/*T1w.nii.gz" \
  --include "sub-09/anat/*T1w.nii.gz" \
  --include "sub-10/anat/*T1w.nii.gz" \
  --include "sub-11/anat/*T1w.nii.gz" \
  --include "sub-12/anat/*T1w.nii.gz" \
  --include "sub-13/anat/*T1w.nii.gz" \
  --include "sub-14/anat/*T1w.nii.gz" \
  --include "sub-15/anat/*T1w.nii.gz" \
  --include "sub-16/anat/*T1w.nii.gz" \
  --include "sub-17/anat/*T1w.nii.gz" \
  --include "sub-18/anat/*T1w.nii.gz" \
  --include "sub-19/anat/*T1w.nii.gz" \
  --include "sub-20/anat/*T1w.nii.gz" 

In [ ]:
!aws s3 sync s3://openneuro.org/ds000114 ./ds000114 \
  --no-sign-request \
  --exclude "*" \
  --include "*/ses-test/anat/*T1w.nii.gz"

In [ ]:
import os

def count_files(label_suffix, search_dirs):
    files = []
    for d in search_dirs:
        for root, dirs, f in os.walk(d):
            for file in f:
                if file.endswith(label_suffix):
                    files.append(os.path.join(root, file))
    print(f"Total: {len(files)}")

count_files("T1w.nii.gz", ["./ds000221", "./ds000114", "./ds002330"])

In [ ]:
!aws s3 sync s3://openneuro.org/ds000221 ./ds000221 \
  --no-sign-request \
  --exclude "*" \
  --include "sub-010002/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010003/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010004/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010005/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010006/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010007/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010010/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010012/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010015/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010016/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010017/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010019/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010020/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010021/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010022/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010023/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010024/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010026/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010027/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010028/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010029/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010030/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010031/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010032/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010033/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010034/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010035/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010036/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010037/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010038/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010039/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010040/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010041/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010042/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010044/ses-01/anat/*T2w.nii.gz" \
  --include "sub-010043/ses-01/anat/*T2w.nii.gz"

In [ ]:
!aws s3 sync s3://openneuro.org/ds002330 ./ds002330 \
  --no-sign-request \
  --exclude "*" \
  --include "sub-01/anat/*T2w.nii.gz" \
  --include "sub-02/anat/*T2w.nii.gz" \
  --include "sub-03/anat/*T2w.nii.gz" \
  --include "sub-04/anat/*T2w.nii.gz" \
  --include "sub-05/anat/*T2w.nii.gz" \
  --include "sub-06/anat/*T2w.nii.gz" \
  --include "sub-07/anat/*T2w.nii.gz" \
  --include "sub-08/anat/*T2w.nii.gz" \
  --include "sub-09/anat/*T2w.nii.gz" \
  --include "sub-10/anat/*T2w.nii.gz" \
  --include "sub-11/anat/*T2w.nii.gz" \
  --include "sub-12/anat/*T2w.nii.gz" \
  --include "sub-13/anat/*T2w.nii.gz" \
  --include "sub-14/anat/*T2w.nii.gz" \
  --include "sub-15/anat/*T2w.nii.gz" 

In [ ]:

count_files("T2w.nii.gz", ["./ds000221", "./ds000114", "./ds002330"])

In [ ]:
!aws s3 sync s3://openneuro.org/ds000221 ./ds000221 \
  --no-sign-request \
  --exclude "*" \
  --include "sub-010002/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010003/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010004/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010005/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010006/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010007/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010010/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010012/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010015/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010016/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010017/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010019/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010020/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010021/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010022/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010023/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010024/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010026/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010027/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010028/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010029/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010030/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010031/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010032/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010033/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010034/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010035/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010036/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010037/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010038/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010039/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010040/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010041/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010042/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010043/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010044/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010045/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010046/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010047/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010048/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010049/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010050/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010051/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010052/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010053/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010056/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010059/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010060/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010061/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010062/ses-01/anat/*FLAIR.nii.gz" \
  --include "sub-010063/ses-01/anat/*FLAIR.nii.gz" 
    

In [ ]:
count_files("FLAIR.nii.gz", ["./ds000221", "./ds000114", "./ds002330"])

In [ ]:
!aws s3 sync s3://openneuro.org/ds000221 ./ds000221 \
  --no-sign-request \
  --exclude "*" \
  --include "sub-010002/ses-01/dwi/*dwi.nii.gz" \
  --include "sub-010003/ses-01/dwi/*dwi.nii.gz" \
  --include "sub-010004/ses-01/dwi/*dwi.nii.gz" \
  --include "sub-010005/ses-01/dwi/*dwi.nii.gz" \
  --include "sub-010006/ses-01/dwi/*dwi.nii.gz" \
  --include "sub-010007/ses-01/dwi/*dwi.nii.gz" \
  --include "sub-010010/ses-01/dwi/*dwi.nii.gz" \
  --include "sub-010012/ses-01/dwi/*dwi.nii.gz" \
  --include "sub-010015/ses-01/dwi/*dwi.nii.gz" \
  --include "sub-010016/ses-01/dwi/*dwi.nii.gz" \
  --include "sub-010017/ses-01/dwi/*dwi.nii.gz" \
  --include "sub-010019/ses-01/dwi/*dwi.nii.gz" \
  --include "sub-010020/ses-01/dwi/*dwi.nii.gz" \
  --include "sub-010021/ses-01/dwi/*dwi.nii.gz" \
  --include "sub-010022/ses-01/dwi/*dwi.nii.gz" \
  --include "sub-010023/ses-01/dwi/*dwi.nii.gz" \
  --include "sub-010024/ses-01/dwi/*dwi.nii.gz" \
  --include "sub-010026/ses-01/dwi/*dwi.nii.gz" \
  --include "sub-010027/ses-01/dwi/*dwi.nii.gz" \
  --include "sub-010028/ses-01/dwi/*dwi.nii.gz" 


In [ ]:
count_files("dwi.nii.gz", ["./ds000221", "./ds000114", "./ds002330"])

In [ ]:
!aws s3 sync s3://openneuro.org/ds002330 ./ds002330 \
  --no-sign-request \
  --exclude "*" \
  --include "sub-01/dwi/*acq-lr_dwi.nii.gz" \
  --include "sub-02/dwi/*acq-lr_dwi.nii.gz" \
  --include "sub-03/dwi/*acq-lr_dwi.nii.gz" \
  --include "sub-04/dwi/*acq-lr_dwi.nii.gz" \
  --include "sub-05/dwi/*acq-lr_dwi.nii.gz" \
  --include "sub-06/dwi/*acq-lr_dwi.nii.gz" \
  --include "sub-07/dwi/*acq-lr_dwi.nii.gz" \
  --include "sub-08/dwi/*acq-lr_dwi.nii.gz" \
  --include "sub-09/dwi/*acq-lr_dwi.nii.gz" \
  --include "sub-10/dwi/*acq-lr_dwi.nii.gz" \
  --include "sub-11/dwi/*acq-lr_dwi.nii.gz" \
  --include "sub-12/dwi/*acq-lr_dwi.nii.gz" \
  --include "sub-13/dwi/*acq-lr_dwi.nii.gz" \
  --include "sub-14/dwi/*acq-lr_dwi.nii.gz" \
  --include "sub-15/dwi/*acq-lr_dwi.nii.gz" \
  --include "sub-16/dwi/*acq-lr_dwi.nii.gz" \
  --include "sub-17/dwi/*acq-lr_dwi.nii.gz" \
  --include "sub-18/dwi/*acq-lr_dwi.nii.gz" \
  --include "sub-19/dwi/*acq-lr_dwi.nii.gz" \
  --include "sub-20/dwi/*acq-lr_dwi.nii.gz" 

In [ ]:
count_files("dwi.nii.gz", ["./ds000221", "./ds000114", "./ds002330"])

In [ ]:
!aws s3 sync s3://openneuro.org/ds000114 ./ds000114 \
  --no-sign-request \
  --exclude "*" \
  --include "*/ses-test/dwi/*dwi.nii.gz"

In [ ]:
count_files("dwi.nii.gz", ["./ds000221", "./ds000114", "./ds002330"])

In [ ]:
!aws s3 sync s3://openneuro.org/ds000221 ./ds000221 \
  --no-sign-request \
  --exclude "*" \
  --include "sub-010002/ses-01/func/*bold.nii.gz" \
  --include "sub-010003/ses-01/func/*bold.nii.gz" \
  --include "sub-010004/ses-01/func/*bold.nii.gz" \
  --include "sub-010005/ses-01/func/*bold.nii.gz" \
  --include "sub-010006/ses-01/func/*bold.nii.gz" \
  --include "sub-010007/ses-01/func/*bold.nii.gz" \
  --include "sub-010010/ses-01/func/*bold.nii.gz" \
  --include "sub-010012/ses-01/func/*bold.nii.gz" \
  --include "sub-010015/ses-01/func/*bold.nii.gz" \
  --include "sub-010016/ses-01/func/*bold.nii.gz" \
  --include "sub-010017/ses-01/func/*bold.nii.gz" \
  --include "sub-010019/ses-01/func/*bold.nii.gz" \
  --include "sub-010020/ses-01/func/*bold.nii.gz" \
  --include "sub-010021/ses-01/func/*bold.nii.gz" \
  --include "sub-010022/ses-01/func/*bold.nii.gz" \
  --include "sub-010023/ses-01/func/*bold.nii.gz" \
  --include "sub-010024/ses-01/func/*bold.nii.gz" \
  --include "sub-010026/ses-01/func/*bold.nii.gz" \
  --include "sub-010027/ses-01/func/*bold.nii.gz" \
  --include "sub-010028/ses-01/func/*bold.nii.gz" 

In [ ]:
!aws s3 sync s3://openneuro.org/ds002330 ./ds002330 \
  --no-sign-request \
  --exclude "*" \
  --include "sub-01/func/*run-01_bold.nii.gz" \
  --include "sub-02/func/*run-01_bold.nii.gz" \
  --include "sub-03/func/*run-01_bold.nii.gz" \
  --include "sub-04/func/*run-01_bold.nii.gz" \
  --include "sub-05/func/*run-01_bold.nii.gz" \
  --include "sub-06/func/*run-01_bold.nii.gz" \
  --include "sub-07/func/*run-01_bold.nii.gz" \
  --include "sub-08/func/*run-01_bold.nii.gz" \
  --include "sub-09/func/*run-01_bold.nii.gz" \
  --include "sub-10/func/*run-01_bold.nii.gz" \
  --include "sub-11/func/*run-01_bold.nii.gz" \
  --include "sub-12/func/*run-01_bold.nii.gz" \
  --include "sub-13/func/*run-01_bold.nii.gz" \
  --include "sub-14/func/*run-01_bold.nii.gz" \
  --include "sub-15/func/*run-01_bold.nii.gz" \
  --include "sub-16/func/*run-01_bold.nii.gz" \
  --include "sub-17/func/*run-01_bold.nii.gz" \
  --include "sub-18/func/*run-01_bold.nii.gz" \
  --include "sub-19/func/*run-01_bold.nii.gz" \
  --include "sub-20/func/*run-01_bold.nii.gz" 

In [ ]:
!aws s3 sync s3://openneuro.org/ds000114 ./ds000114 \
  --no-sign-request \
  --exclude "*" \
  --include "*/ses-test/func/*task-overtwordrepetition_bold.nii.gz"

In [ ]:
count_files("bold.nii.gz", ["./ds000221", "./ds000114", "./ds002330"])

## Preprocessing

Each volume is processed as follows:
- 4D inputs (BOLD, DWI) -> take first timepoint
- Extract 5 axial slices at 10/25/50/75/90 percentile depths
- Z-score normalize -> clip ±3σ -> scale to uint8
- Save as RGB PNG

Raw NIfTI files are deleted after slicing to save disk space.

In [ ]:

import nibabel as nib
import numpy as np
from PIL import Image

def zscore_normalize(slice_2d):
    mean = slice_2d.mean()
    std = slice_2d.std()
    if std == 0:
        return slice_2d
    return (slice_2d - mean) / std

def normalize_to_uint8(slice_2d):
    slice_2d = np.clip(slice_2d, -3, 3)  # clip z-score outliers
    slice_2d = (slice_2d + 3) / 6       # scale to 0-1
    return (slice_2d * 255).astype(np.uint8)

def extract_slices(nifti_path, output_dir, subject_id, label):
    img = nib.load(nifti_path)
    data = img.get_fdata()
    
    # if 4D (bold/dwi), take first volume
    if data.ndim == 4:
        data = data[..., 0]
    
    # get slices at 10/25/50/75/90 percentile
    z = data.shape[2]
    percentiles = [0.10, 0.25, 0.50, 0.75, 0.90]
    indices = [int(p * z) for p in percentiles]
    
    for i, idx in enumerate(indices):
        slice_2d = data[:, :, idx]
        slice_2d = zscore_normalize(slice_2d)
        slice_2d = normalize_to_uint8(slice_2d)
        
        img_pil = Image.fromarray(slice_2d).convert("RGB")
        fname = f"{subject_id}_slice{i}.png"
        img_pil.save(os.path.join(output_dir, fname))

def process_label(label, suffix, dataset_dirs, output_base):
    output_dir = os.path.join(output_base, label)
    os.makedirs(output_dir, exist_ok=True)
    
    count = 0
    for dataset_path in dataset_dirs:
        dataset_id = os.path.basename(dataset_path)
        for root, dirs, files in os.walk(dataset_path):
            for f in files:
                if f.endswith(suffix):
                    filepath = os.path.join(root, f)
                    parts = filepath.split(os.sep)
                    subject = next(p for p in parts if p.startswith("sub-"))
                    subject_id = f"{dataset_id}_{subject}"
                    
                    try:
                        extract_slices(filepath, output_dir, subject_id, label)
                        count += 1
                        print(f" [OK] {subject_id} ({count})")
                        os.remove(filepath) 
                    except Exception as e:
                        print(f" [FAILED] {subject_id} — {e}")
    
    print(f"\n{label} done: {count} volumes → {count*5} slices\n")

#one label at a time
dataset_dirs = ["./ds000221", "./ds000114", "./ds002330"]
output_base = "/kaggle/working/processed"

process_label("t1w",   "T1w.nii.gz",   dataset_dirs, output_base)
process_label("t2w",   "T2w.nii.gz",   dataset_dirs, output_base)
process_label("flair", "FLAIR.nii.gz", dataset_dirs, output_base)
process_label("dwi",   "dwi.nii.gz",   dataset_dirs, output_base)
process_label("bold",  "bold.nii.gz",  dataset_dirs, output_base)